# AgentInjectionBench — Dataset Generation on Kaggle

Runs the full generation pipeline using Ollama + qwen2.5:14b on Kaggle's T4 GPUs.
**Expected output: ~3600 raw samples → 2500+ curated.**

Runtime: ~2-4 hours. Make sure GPU is enabled (Settings → Accelerator → GPU T4 x2).

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install dependencies + Ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
# Step 3: Start Ollama server in background
import subprocess, time
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started')

In [ ]:
# Step 4: Pull model — qwen2.5:14b on dual T4 (30GB VRAM)
!ollama pull qwen2.5:14b

In [ ]:
# Step 5: Clone the repo
!git clone https://github.com/ppradyoth/AgentInjectionBench.git
%cd AgentInjectionBench
!pip install pyyaml -q

In [ ]:
# Step 6: Quick connectivity check
import urllib.request, json
resp = urllib.request.urlopen('http://localhost:11434/api/tags')
data = json.loads(resp.read())
print('Models available:', [m['name'] for m in data.get('models', [])])

In [ ]:
# Step 7: Run generation — 30 variations × 120 seeds = ~3600 raw samples
# This is the long step (~2-4 hours). Progress logged every seed.
!python3 -m generation.generate \
    --provider ollama \
    --model qwen2.5:14b \
    --variations 30 \
    --temperature 0.85 \
    --output data/agent_injection_bench_raw.jsonl

In [ ]:
# Step 8: How many raw samples did we get?
!wc -l data/agent_injection_bench_raw.jsonl

In [ ]:
# Step 9: Curate + deduplicate + split
!python3 -m generation.curate \
    --input data/agent_injection_bench_raw.jsonl \
    --output data/agent_injection_bench.jsonl \
    --split

In [ ]:
# Step 10: Validate schema
!python3 -m generation.validate_schema data/agent_injection_bench.jsonl

In [ ]:
# Step 11: Full statistics
!python3 -m generation.stats --input data/agent_injection_bench.jsonl

In [ ]:
# Step 12: Package output files for download
!tar -czf aib_dataset_v0.2.tar.gz \
    data/agent_injection_bench.jsonl \
    data/agent_injection_bench_raw.jsonl \
    data/splits/
print('Download aib_dataset_v0.2.tar.gz from the output panel on the right.')

## After downloading

On your local machine:

```bash
cd AgentInjectionBench
tar -xzf aib_dataset_v0.2.tar.gz

# Validate locally
python3 generation/validate_schema.py data/agent_injection_bench.jsonl

# Push to all 3
git add data/
git commit -m "Expand dataset to 2500+ samples via qwen2.5:14b on Kaggle"
git push origin main
git push hf-dataset main
cd /tmp/aib-space && cp ~/path/to/data/agent_injection_bench.jsonl data/ && git add data/ && git commit -m "Update dataset" && git push origin main
```